In [42]:
# =============================================================================
# Monthly Feature Matrix Preparation — Product B
# =============================================================================
# Produces two dataframes:
#   training_df  — station pixels × months, with insitu_monthly_total (train stations)
#   test_df      — station pixels × months, with insitu_monthly_total (test stations)
#
# One row = one station × one year-month.
#
# Key differences from daily pipeline:
#   - Target: insitu_monthly_total (sum of valid daily obs per station-month)
#   - Dynamic features: monthly aggregate rasters from monthly_raster_aggregation.py
#   - Dropped: kriged ERA5L predictor, intensity class, season column
#   - Added: month_sin, month_cos (cyclical temporal encoding)
#   - Completeness filter: station-months with < MIN_VALID_DAYS excluded from target
# =============================================================================
 
import os
import gc
import glob
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.spatial import cKDTree
from tqdm import tqdm
 
# ── Configuration ─────────────────────────────────────────────────────────────
START_YEAR, START_MONTH = 2001, 1
END_YEAR,   END_MONTH   = 2005, 12
 
# Minimum valid daily observations required to include a station-month as a
# training target. Station-months below this threshold are excluded entirely
# (not imputed). 20/28-31 ≈ ~70% completeness floor.
MIN_VALID_DAYS = 20
 
# Paths — monthly raster aggregation outputs
MONTHLY_RASTER_DIR = r"..\Data\Processed\RegressionModel\Monthly_Rasters"
TEMPLATE_PATH      = r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif"
 
# Static raster paths (same as daily pipeline)
STATIC_PATHS = {
    "cluster":         r"..\Data\Processed\Spatial_clustering\Precipitation_clusters.tif",
    "elev_mean":       r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_mean.tif",
    "elev_std":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_std.tif",
    "elev_max":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif",
    "elev_min":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_min.tif",
    "slope_mean":      r"..\Data\Datasets\DEM_1km\DEM_1km_slope_mean.tif",
    "aspect_mean":     r"..\Data\Datasets\DEM_1km\DEM_1km_aspect_mean.tif",
    "elev_smoothed":   r"..\Data\Processed\Fetch\Preprocessing\dem_smoothed.tif",
    "fetch_1000m_dist":r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_1000m.tif",
    "fetch_1000m_dir": r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_1000m.tif",
    "fetch_500m_dist": r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_500m.tif",
    "fetch_500m_dir":  r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_500m.tif",
    "local_relief":    r"..\Data\Processed\Fetch\Preprocessing\local_relief.tif",
}
 
# In-situ
INSITU_CSV      = r"..\Data\Processed\Insitu\insitu_rainfall_1991_2020.csv"
TRAIN_STATIONS  = r"..\Data\Processed\Split_stations\train_stations.csv"
TEST_STATIONS   = r"..\Data\Processed\Split_stations\test_stations.csv"
 
# Output directories
OUT_BASE           = r"..\Data\Processed\RegressionModel"
MONTHLY_TRAIN_DIR  = os.path.join(OUT_BASE, "monthly_dynamic_train_df")
MONTHLY_TEST_DIR   = os.path.join(OUT_BASE, "monthly_dynamic_test_df")
PARQUET_TRAIN_DIR  = os.path.join(OUT_BASE, "yearly_parquet_train")
PARQUET_TEST_DIR   = os.path.join(OUT_BASE, "yearly_parquet_test")
STATIC_DIR         = os.path.join(OUT_BASE, "static_df")
 
for d in [MONTHLY_TRAIN_DIR, MONTHLY_TEST_DIR, PARQUET_TRAIN_DIR,
          PARQUET_TEST_DIR, STATIC_DIR, OUT_BASE]:
    os.makedirs(d, exist_ok=True)
 
# ── Template ──────────────────────────────────────────────────────────────────
 
template = rxr.open_rasterio(TEMPLATE_PATH, masked=True)

In [43]:
# ── Helper: cyclical month encoding ──────────────────────────────────────────
 
def month_cyclical(month: int):
    """Return (sin, cos) encoding for a calendar month (1–12)."""
    angle = 2 * np.pi * month / 12
    return np.sin(angle), np.cos(angle)
 
# ── Helper: month label ───────────────────────────────────────────────────────
 
def mlabel(year: int, month: int) -> str:
    return f"{year}{month:02d}"
 
# ── 1. Static feature dataframe ───────────────────────────────────────────────
# Built once, reused for every station-month row.
 
static_csv = os.path.join(STATIC_DIR, "static_df.csv")
 
if os.path.exists(static_csv):
    print("Loading cached static df...")
    static_df = pd.read_csv(static_csv)
else:
    print("Building static df from rasters...")
    aligned = []
    for name, path in STATIC_PATHS.items():
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        aligned.append(da)
    static_xr = xr.merge(aligned, compat="override")
    static_df  = static_xr.to_dataframe().reset_index()
    static_df.to_csv(static_csv, index=False)
    del aligned, static_xr
    gc.collect()
 
static_df[["x", "y"]] = static_df[["x", "y"]].round(5)
print(f"Static df: {len(static_df):,} pixels")

Loading cached static df...
Static df: 484,619 pixels


In [44]:
# ── 2. Monthly insitu totals with completeness filter ─────────────────────────
# Aggregate daily gauge observations to monthly totals.
# Exclude station-months where valid daily count < MIN_VALID_DAYS.
# This prevents low-coverage months from acting as training targets.
 
print("\nAggregating insitu to monthly totals...")
 
insitu_raw = pd.read_csv(INSITU_CSV)
insitu_raw["date"]  = pd.to_datetime(insitu_raw[["year", "month", "day"]])
insitu_raw          = insitu_raw.rename(columns={"lat": "y", "long": "x",
                                                  "rainfall_sum": "insitu_prec"})
insitu_raw          = insitu_raw[insitu_raw["insitu_prec"] >= 0]   # drop sentinel negatives
 
# Count valid days and sum per station-month
insitu_monthly = (
    insitu_raw
    .groupby(["index_no", "y", "x", "year", "month"])
    .agg(
        insitu_monthly_total=("insitu_prec", "sum"),
        valid_days          =("insitu_prec", "count")
    )
    .reset_index()
)
 
# Apply completeness filter
n_before = len(insitu_monthly)
insitu_monthly = insitu_monthly[insitu_monthly["valid_days"] >= MIN_VALID_DAYS].copy()
n_after  = len(insitu_monthly)
print(f"  Station-months before completeness filter: {n_before:,}")
print(f"  Station-months after  (>= {MIN_VALID_DAYS} valid days): {n_after:,}")
print(f"  Excluded: {n_before - n_after:,} ({100*(n_before-n_after)/n_before:.1f}%)")
 
# Filter to analysis period
insitu_monthly = insitu_monthly[
    (insitu_monthly["year"]  >= START_YEAR) &
    (insitu_monthly["year"]  <= END_YEAR)
].copy()
insitu_monthly[["x", "y"]] = insitu_monthly[["x", "y"]].round(5)
 
# Train / test split
train_ids = set(pd.read_csv(TRAIN_STATIONS)["index_no"])
test_ids  = set(pd.read_csv(TEST_STATIONS )["index_no"])
 
insitu_train_monthly = insitu_monthly[insitu_monthly["index_no"].isin(train_ids)].copy()
insitu_test_monthly  = insitu_monthly[insitu_monthly["index_no"].isin(test_ids) ].copy()
 
print(f"\n  Train station-months: {len(insitu_train_monthly):,}")
print(f"  Test  station-months: {len(insitu_test_monthly):,}")


Aggregating insitu to monthly totals...
  Station-months before completeness filter: 23,456
  Station-months after  (>= 20 valid days): 23,356
  Excluded: 100 (0.4%)

  Train station-months: 2,886
  Test  station-months: 1,035


In [ ]:
# ── 3. Monthly raster path dictionary ─────────────────────────────────────────
# Maps (year, month, variable_name) → filepath.
 
def build_monthly_paths(year: int, month: int) -> dict:
    """
    Return dict of {feature_name: filepath} for all monthly rasters
    for the given year-month. Missing files are noted but not fatal —
    the loader handles absent paths with NaN fill.
    """
    L = mlabel(year, month)
    R = MONTHLY_RASTER_DIR
    return {
        # Precipitation sums — all five products
        "era5l_monthly_sum":        f"{R}/era5l/era5l_monthly_sum_{L}.tif",
        "chirps_monthly_sum":       f"{R}/chirps/chirps_monthly_sum_{L}.tif",
        "har_monthly_sum":          f"{R}/har/har_monthly_sum_{L}.tif",
        "persiann_monthly_sum":     f"{R}/persiann/persiann_monthly_sum_{L}.tif",
        "mswep_monthly_sum":        f"{R}/mswep/mswep_monthly_sum_{L}.tif",
        
        # Secondary precipitation variables from CHIRPS
        "chirps_wet_day_count":      f"{R}/chirps/chirps_wet_day_count_{L}.tif",
        "chirps_monthly_std":        f"{R}/chirps/chirps_monthly_std_{L}.tif",

        #GLDAS weather variables
        'gldas_air_temp':   f"{R}/gldas_air_temp/gldas_air_temp_monthly_mean_{L}.tif",
        'gldas_lwdown': f"{R}/gldas_lwdown/gldas_lwdown_monthly_sum_{L}.tif",
        'gldas_specific_humidity': f"{R}/gldas_specific_humidity/gldas_specific_humidity_monthly_mean_{L}.tif",
        'gldas_surface_pressure': f"{R}/gldas_surface_pressure/gldas_surface_pressure_monthly_mean_{L}.tif",
        'gldas_swdown': f"{R}/gldas_swdown/gldas_swdown_monthly_sum_{L}.tif",

        # MODIS EVI
        "evi_monthly_mean":         f"{R}/modis_evi/evi_monthly_mean_{L}.tif",
        "evi_missing_fraction":     f"{R}/modis_evi/evi_missing_fraction_{L}.tif",

        # MODIS LST
        "lst_d_monthly_mean":       f"{R}/modis_lst/lst_d_monthly_mean_{L}.tif",
        "lst_d_missing_fraction":   f"{R}/modis_lst/lst_d_missing_fraction_{L}.tif",
        "lst_n_monthly_mean":       f"{R}/modis_lst/lst_n_monthly_mean_{L}.tif",
        "lst_n_missing_fraction":   f"{R}/modis_lst/lst_n_missing_fraction_{L}.tif",
        "lst_diff_monthly_mean":    f"{R}/modis_lst/lst_diff_monthly_mean_{L}.tif",
        "lst_diff_missing_fraction":f"{R}/modis_lst/lst_diff_missing_fraction_{L}.tif"

        #Kriged Insitu data
#        "insitu_kriged":   f"{R}/Insitu_kriged/insitu_precip_{L}.tif"
    }

In [46]:
# ── 4. Main month loop ────────────────────────────────────────────────────────
 
month_range = pd.period_range(
    start=f"{START_YEAR}-{START_MONTH:02d}",
    end  =f"{END_YEAR}-{END_MONTH:02d}",
    freq ="M"
)
 
print(f"\nProcessing {len(month_range)} months...\n")
 
for period in tqdm(month_range, desc="Monthly feature extraction"):
    year  = period.year
    month = period.month
    label = mlabel(year, month)
 
    train_out = os.path.join(MONTHLY_TRAIN_DIR, f"merged_{label}.csv")
    test_out  = os.path.join(MONTHLY_TEST_DIR,  f"merged_{label}.csv")
 
    if os.path.exists(train_out) and os.path.exists(test_out):
        continue   # skip if already processed
 
    # ── 4a. Load monthly rasters and flatten to pixel df ──────────────────
 
    paths = build_monthly_paths(year, month)
    aligned = []
    missing_vars = []
 
    for name, path in paths.items():
        if not os.path.exists(path):
            missing_vars.append(name)
            continue
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        aligned.append(da.load())
        da.close()
 
    if missing_vars:
        tqdm.write(f"  [{label}] missing rasters: {missing_vars}")
 
    if not aligned:
        tqdm.write(f"  [{label}] no rasters found — skipping")
        continue
 
    # Merge all dynamic rasters → pixel dataframe
    dynamic_xr = xr.merge(aligned, compat="override")
    dynamic_df = dynamic_xr.to_dataframe().reset_index()
    dynamic_df[["x", "y"]] = dynamic_df[["x", "y"]].round(5)
 
    # Any variable that had no raster gets a NaN column
    for name in missing_vars:
        dynamic_df[name] = np.nan
 
    del aligned, dynamic_xr
    gc.collect()
 
    # ── 4b. Merge with static features ──────────────────────────────────── 
    monthly_grid = dynamic_df.merge(static_df, on=["x", "y"], how="inner")
 
    # Add temporal features — cyclical month encoding
    sin_m, cos_m = month_cyclical(month)
    monthly_grid["month_sin"] = sin_m
    monthly_grid["month_cos"] = cos_m
    monthly_grid["year"]  = year
    monthly_grid["month"] = month
 
    del dynamic_df
    gc.collect()
 
    # ── 4c. cKDTree match to station locations ─────────────────────────────
    # Same spatial matching logic as daily pipeline.
    # Threshold 0.006° ≈ ~600m at Nepal latitudes — one pixel tolerance.
 
    insitu_train_m = insitu_train_monthly[
        (insitu_train_monthly["year"]  == year) &
        (insitu_train_monthly["month"] == month)
    ].copy()
 
    insitu_test_m = insitu_test_monthly[
        (insitu_test_monthly["year"]  == year) &
        (insitu_test_monthly["month"] == month)
    ].copy()
 
    if insitu_train_m.empty and insitu_test_m.empty:
        del monthly_grid
        gc.collect()
        continue
 
    tree = cKDTree(monthly_grid[["x", "y"]].values)
 
    # ── Train stations ──────────────────────────────────────────────────
 
    if not insitu_train_m.empty:
        dist_tr, idx_tr = tree.query(insitu_train_m[["x", "y"]].values, k=1)
        mask_tr = dist_tr <= 0.006
 
        if mask_tr.any():
            station_rows = insitu_train_m.iloc[mask_tr].reset_index(drop=True)
            grid_rows    = monthly_grid.iloc[idx_tr[mask_tr]].reset_index(drop=True)
 
            final_train = pd.concat([
                station_rows[["index_no", "y", "x", "year", "month",
                               "insitu_monthly_total", "valid_days"]],
                grid_rows.drop(columns=["x", "y", "year", "month"])
            ], axis=1)
 
            final_train.to_csv(train_out, index=False)
 
    # ── Test stations ───────────────────────────────────────────────────
 
    if not insitu_test_m.empty:
        dist_te, idx_te = tree.query(insitu_test_m[["x", "y"]].values, k=1)
        mask_te = dist_te <= 0.006
 
        if mask_te.any():
            station_rows = insitu_test_m.iloc[mask_te].reset_index(drop=True)
            grid_rows    = monthly_grid.iloc[idx_te[mask_te]].reset_index(drop=True)
 
            final_test = pd.concat([
                station_rows[["index_no", "y", "x", "year", "month",
                               "insitu_monthly_total", "valid_days"]],
                grid_rows.drop(columns=["x", "y", "year", "month"])
            ], axis=1)
 
            final_test.to_csv(test_out, index=False)
 
    del monthly_grid, insitu_train_m, insitu_test_m
    gc.collect()
 


Processing 60 months...



Monthly feature extraction: 100%|██████████| 60/60 [01:20<00:00,  1.34s/it]


In [47]:
# ── 5. Aggregate monthly CSVs to yearly parquet → final combined df ───────────
 
def aggregate_to_parquet(monthly_dir: str, parquet_dir: str, split_label: str) -> str:
    """
    Read monthly CSVs year by year, write to parquet, then concatenate
    all years into a single final df. Returns path to final parquet.
    """
    parquet_paths = []
 
    for year in range(START_YEAR, END_YEAR + 1):
        yearly_path = os.path.join(parquet_dir, f"full_{year}.parquet")
        monthly_periods = pd.period_range(
            start=f"{year}-01", end=f"{year}-12", freq="M"
        )
        expected = [
            os.path.join(monthly_dir, f"merged_{mlabel(p.year, p.month)}.csv")
            for p in monthly_periods
        ]
        found = [f for f in expected if os.path.exists(f)]
 
        if not found:
            print(f"  [{split_label}] {year} — no monthly files found, skipping")
            continue
 
        writer = None
        for fpath in found:
            chunk = pd.read_csv(fpath)
            table = pa.Table.from_pandas(chunk)
            if writer is None:
                writer = pq.ParquetWriter(yearly_path, table.schema)
            writer.write_table(table)
 
        if writer:
            writer.close()
            parquet_paths.append(yearly_path)
            print(f"  [{split_label}] {year} — {len(found)} months written to parquet")
 
    if not parquet_paths:
        print(f"  [{split_label}] No data — final df not written")
        return None
 
    final_df   = pd.concat([pd.read_parquet(p) for p in parquet_paths], ignore_index=True)
    final_path = os.path.join(OUT_BASE, f"final_{split_label}_df.parquet")
    final_df.to_parquet(final_path, index=False)
 
    # Lightweight CSV summary for quick inspection (first 10k rows)
    final_df.head(10_000).to_csv(
        os.path.join(OUT_BASE, f"final_{split_label}_df_preview.csv"), index=False
    )
 
    print(f"\n  [{split_label}] Final df: {len(final_df):,} rows × {len(final_df.columns)} cols")
    print(f"  Saved: {final_path}")
    return final_path
 
 
print("\n" + "=" * 60)
print("Aggregating to yearly parquet files...")
print("=" * 60)
 
train_final_path = aggregate_to_parquet(MONTHLY_TRAIN_DIR, PARQUET_TRAIN_DIR, "train")
test_final_path  = aggregate_to_parquet(MONTHLY_TEST_DIR,  PARQUET_TEST_DIR,  "test")


Aggregating to yearly parquet files...
  [train] 2001 — 12 months written to parquet
  [train] 2002 — 12 months written to parquet
  [train] 2003 — 12 months written to parquet
  [train] 2004 — 12 months written to parquet
  [train] 2005 — 12 months written to parquet

  [train] Final df: 2,886 rows × 45 cols
  Saved: ..\Data\Processed\RegressionModel\final_train_df.parquet
  [test] 2001 — 12 months written to parquet
  [test] 2002 — 12 months written to parquet
  [test] 2003 — 12 months written to parquet
  [test] 2004 — 12 months written to parquet
  [test] 2005 — 12 months written to parquet

  [test] Final df: 1,035 rows × 45 cols
  Saved: ..\Data\Processed\RegressionModel\final_test_df.parquet


In [48]:
# ── 6. Sanity check on final training df ──────────────────────────────────────
 
if train_final_path:
    df = pd.read_parquet(train_final_path)
 
    print("\n" + "=" * 60)
    print("Training df sanity check")
    print("=" * 60)
    print(f"Shape              : {df.shape}")
    print(f"Columns            : {list(df.columns)}")
    print(f"\nStation-months     : {len(df):,}")
    print(f"Unique stations    : {df['index_no'].nunique()}")
    print(f"Year range         : {df['year'].min()}–{df['year'].max()}")
    print(f"Month distribution :\n{df['month'].value_counts().sort_index()}")
    print(f"\nCluster distribution:\n{df['cluster'].value_counts().sort_index()}")
    print(f"\nTarget (insitu_monthly_total) summary:")
    print(df["insitu_monthly_total"].describe().round(2))
    print(f"\nValid days distribution (completeness filter applied at >= {MIN_VALID_DAYS}):")
    print(df["valid_days"].describe().round(1))
 
    # Flag any rows where all precipitation sums are NaN
    prec_cols = [c for c in df.columns if "monthly_sum" in c]
    all_prec_nan = df[prec_cols].isna().all(axis=1).sum()
    if all_prec_nan > 0:
        print(f"\nWARNING: {all_prec_nan} rows have NaN in all precipitation sum columns")
        print("  → Check that monthly raster files exist for those year-months")
 
    print("\nDone.")


Training df sanity check
Shape              : (2886, 45)
Columns            : ['index_no', 'y', 'x', 'year', 'month', 'insitu_monthly_total', 'valid_days', 'spatial_ref_x', 'era5l_monthly_sum', 'chirps_monthly_sum', 'har_monthly_sum', 'persiann_monthly_sum', 'mswep_monthly_sum', 'chirps_wet_day_count', 'chirps_monthly_std', 'gldas_air_temp', 'gldas_lwdown', 'gldas_specific_humidity', 'gldas_surface_pressure', 'gldas_swdown', 'evi_monthly_mean', 'evi_missing_fraction', 'lst_d_monthly_mean', 'lst_d_missing_fraction', 'lst_n_monthly_mean', 'lst_n_missing_fraction', 'lst_diff_monthly_mean', 'lst_diff_missing_fraction', 'insitu_kriged', 'spatial_ref_y', 'cluster', 'elev_mean', 'elev_std', 'elev_max', 'elev_min', 'slope_mean', 'aspect_mean', 'elev_smoothed', 'fetch_1000m_dist', 'fetch_1000m_dir', 'fetch_500m_dist', 'fetch_500m_dir', 'local_relief', 'month_sin', 'month_cos']

Station-months     : 2,886
Unique stations    : 50
Year range         : 2001–2005
Month distribution :
month
1     24

In [49]:
# ── 7. Clean up intermediate files ──────────────────────────────────────────────

import shutil

print("\n" + "=" * 60)
print("Cleaning up intermediate files and directories...")
print("=" * 60)

dirs_to_remove = [
    MONTHLY_TRAIN_DIR,
    MONTHLY_TEST_DIR,
    PARQUET_TRAIN_DIR,
    PARQUET_TEST_DIR,
    STATIC_DIR
]

for d in dirs_to_remove:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed directory: {d}")

# # Also remove preview csvs
# for split in ['train', 'test']:
#     preview_csv = os.path.join(OUT_BASE, f"final_{split}_df_preview.csv")
#     if os.path.exists(preview_csv):
#         os.remove(preview_csv)
#         print(f"Removed preview CSV: {preview_csv}")

print("\nCleanup complete.")



Cleaning up intermediate files and directories...
Removed directory: ..\Data\Processed\RegressionModel\monthly_dynamic_train_df
Removed directory: ..\Data\Processed\RegressionModel\monthly_dynamic_test_df
Removed directory: ..\Data\Processed\RegressionModel\yearly_parquet_train
Removed directory: ..\Data\Processed\RegressionModel\yearly_parquet_test
Removed directory: ..\Data\Processed\RegressionModel\static_df

Cleanup complete.
